# SUBIT-LUCA: PBMC 3k Demo

This notebook demonstrates the core workflow of SUBIT-LUCA:
1. Load human PBMC single‑cell RNA‑seq data (10x Genomics)
2. Encode each cell into a 64‑state probability distribution (P(SUBIT))
3. Compute metrics: entropy, LUCA distance
4. Visualise the results in UMAP space

The goal is to see if SUBIT distributions capture meaningful biological variation.

## 1. Setup and imports

In [ ]:
import sys
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

# Set paths if needed (assuming the repo root is in PYTHONPATH)
sys.path.insert(0, '..')

from datasets.human import load_human_pbmc, load_human_pbmc_with_cycle_scores
from encoder.universal import UniversalEncoder
from metrics.entropy import entropy, normalized_entropy
from metrics.luca import luca_distance, luca_projection
from levels.level0_luca import Level0_LUCA

# Optional: for nicer plots
sc.set_figure_params(dpi=100, figsize=(6, 5))

## 2. Load PBMC data and add cell cycle scores (for validation)

In [ ]:
# Load data with cell cycle scores (S and G2M)
adata = load_human_pbmc_with_cycle_scores()
print(f"Data loaded: {adata.n_obs} cells, {adata.n_vars} genes")
print("Cell cycle phase distribution:\n", adata.obs['phase'].value_counts())

## 3. Encode to SUBIT 64-state distribution

In [ ]:
# Create encoder (will normalise and log1p by default)
encoder = UniversalEncoder(normalize=True)
P64 = encoder.encode(adata)
adata.obsm['X_subit64'] = P64
print(f"P64 shape: {P64.shape}")
print("Example row (first cell) - first 10 entries:\n", P64[0, :10])

## 4. Compute metrics

In [ ]:
# Shannon entropy (nats) – measure of uncertainty/plasticity
adata.obs['subit_entropy'] = entropy(P64, base=np.e)
adata.obs['subit_entropy_norm'] = normalized_entropy(P64, base=np.e)

# LUCA distance (expected popcount) – evolutionary complexity
adata.obs['luca_distance'] = luca_distance(P64)

# LUCA projection (mass on LUCA-allowed states)
luca_level = Level0_LUCA()
adata.obs['luca_mass'] = luca_projection(P64, luca_level.allowed_states())

# Combined complexity (as defined earlier)
adata.obs['complexity'] = adata.obs['subit_entropy'] + adata.obs['luca_distance']

print("Metrics computed:")
adata.obs[['subit_entropy', 'luca_distance', 'complexity', 'luca_mass']].describe()

## 5. Dimensionality reduction and visualisation

In [ ]:
# UMAP on raw gene expression (for reference)
sc.pp.pca(adata)
sc.pp.neighbors(adata, n_pcs=30)
sc.tl.umap(adata, random_state=42)

# UMAP on SUBIT space (using the 64-state probabilities as features)
sc.pp.neighbors(adata, use_rep='X_subit64', n_pcs=None)  # use full 64D
sc.tl.umap(adata, random_state=42, key_added='X_umap_subit')

# Store the normal UMAP for later comparison
adata.obsm['X_umap_gene'] = adata.obsm['X_umap'].copy()

In [ ]:
# Plot UMAP coloured by cell cycle phase (ground truth for WHEN)
sc.pl.umap(adata, color='phase', title='UMAP (gene space) – cell cycle phase', show=True)

In [ ]:
# Plot UMAP coloured by SUBIT metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
sc.pl.umap(adata, color='subit_entropy', ax=axes[0,0], show=False, title='Entropy (SUBIT)')
sc.pl.umap(adata, color='luca_distance', ax=axes[0,1], show=False, title='LUCA distance')
sc.pl.umap(adata, color='complexity', ax=axes[1,0], show=False, title='Complexity')
sc.pl.umap(adata, color='luca_mass', ax=axes[1,1], show=False, title='LUCA mass')
plt.tight_layout()
plt.show()

## 6. Compare SUBIT-space UMAP with gene-space UMAP

In [ ]:
# Switch to SUBIT-space UMAP coordinates
adata.obsm['X_umap'] = adata.obsm['X_umap_subit']
sc.pl.umap(adata, color='phase', title='UMAP (SUBIT space) – cell cycle phase', show=True)
# Restore original UMAP
adata.obsm['X_umap'] = adata.obsm['X_umap_gene']

## 7. Examine which SUBIT states dominate

In [ ]:
# Find the most probable state for each cell
most_probable_state_idx = np.argmax(P64, axis=1)
adata.obs['dominant_state'] = most_probable_state_idx

# Map index to human-readable string
from subit.state import STATE_NAMES
adata.obs['dominant_state_name'] = [STATE_NAMES[i] for i in most_probable_state_idx]

# Show top 10 most common dominant states
print("Top 10 dominant SUBIT states:\n", adata.obs['dominant_state_name'].value_counts().head(10))

In [ ]:
# Cross-tabulation with cell cycle phase
cross = pd.crosstab(adata.obs['phase'], adata.obs['dominant_state_name'], normalize='index')
# Show only states with >5% in any phase
cross = cross.loc[:, cross.max() > 0.05]
cross.round(3)

## 8. Conclusion

This notebook demonstrated:
- Encoding scRNA‑seq data into a 64‑state SUBIT distribution
- Computing biologically interpretable metrics (entropy, LUCA distance, complexity)
- Visualising cells in gene and SUBIT latent spaces
- Linking dominant SUBIT states to cell cycle phase

Next steps: apply the same pipeline to bacterial datasets and create the universal map of life.